In [8]:
# 01_setup.py
import sys
import subprocess

print("🔧 Setting up environment...")

# Install required packages (run once)
packages = ["biopython", "pandas", "numpy", "scikit-learn", "matplotlib", "seaborn", "tqdm"]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"✅ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
from Bio import Entrez
print("\n✅ Environment ready!")
print("Next: Run 02_model_definition.py")

🔧 Setting up environment...
Installing biopython...
✅ pandas already installed
✅ numpy already installed
Installing scikit-learn...
✅ matplotlib already installed
✅ seaborn already installed
✅ tqdm already installed

✅ Environment ready!
Next: Run 02_model_definition.py


In [12]:
# 02_model_definition.py
!pip install torch
import torch
import torch.nn as nn
import torch.nn.functional as F
from Bio import Entrez
import os
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ====================== YOUR MODEL CLASSES ======================
class LogDecayMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=8, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = int(expand * d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=4, groups=self.d_inner, padding=3)
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)
        A_init = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A_init))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        batch, seq_len, _ = x.shape
        projected = self.in_proj(x)
        x_inner, res_gate = projected.chunk(2, dim=-1)
        x_conv = self.conv1d(x_inner.transpose(1, 2))[:, :, :seq_len].transpose(1, 2)
        x_act = F.silu(x_conv)
        ssm_params = self.x_proj(x_act)
        dt_raw, B, C = torch.split(ssm_params, [1, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))
        A = -torch.exp(self.A_log.float())
        log_dA = dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0)
        dA_decay = torch.exp(torch.cumsum(log_dA, dim=1))
        dB = dt.unsqueeze(-1) * B.unsqueeze(2)
        state_inputs = dB * x_act.unsqueeze(-1)
        h_state = state_inputs / (dA_decay + 1e-8)
        h_cumsum = torch.cumsum(h_state, dim=1)
        y = (dA_decay * h_cumsum * C.unsqueeze(2)).sum(dim=-1)
        return self.out_proj(y + x_act * self.D * F.silu(res_gate))


class PanGenomicMambaArchitecture(nn.Module):
    def __init__(self, vocab_size=4096, n_pgap_classes=5, d_model=64, n_layers=4):
        super().__init__()
        self.half_dim = d_model // 2
        self.nuc_embedding = nn.Embedding(vocab_size, self.half_dim)
        self.anno_embedding = nn.Embedding(n_pgap_classes, self.half_dim)
        self.mamba_layers = nn.ModuleList([LogDecayMambaBlock(d_model) for _ in range(n_layers)])
        self.layer_norm = nn.LayerNorm(d_model)
        self.syntax_head = nn.Linear(d_model, vocab_size)
        self.annotation_head = nn.Linear(d_model, n_pgap_classes)

    def forward(self, nuc_tokens, anno_tokens, return_embeddings=False):
        n_feats = self.nuc_embedding(nuc_tokens)
        a_feats = self.anno_embedding(anno_tokens)
        
        if a_feats.dim() == 2:
            a_feats = a_feats.unsqueeze(1)
        if a_feats.size(1) != n_feats.size(1):
            a_feats = a_feats.expand(-1, n_feats.size(1), -1)
            
        x = torch.cat([n_feats, a_feats], dim=-1)
        for layer in self.mamba_layers:
            x = layer(x)
        x = self.layer_norm(x)
        latent = x.mean(dim=1)
        
        if return_embeddings:
            return latent
        return self.syntax_head(x).transpose(1, 2), self.annotation_head(latent)


print("✅ Model classes loaded successfully!")
print("Next: Run 03_load_model.py")

Using device: cpu
✅ Model classes loaded successfully!
Next: Run 03_load_model.py



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json

DATA_DIR = os.path.join(os.getcwd(), "data")
MODEL_PATH = os.path.join(DATA_DIR, "spyogenes_mamba_weights.pth")
CONFIG_PATH = os.path.join(DATA_DIR, "mamba_tokenizer_config.json")

print(f"Model path: {MODEL_PATH}")
print(f"Config path: {CONFIG_PATH}")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)
kmer_to_idx = config["kmer_to_idx"]